In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import add_self_loops, degree
import warnings
warnings.filterwarnings('ignore')

In [2]:
dataset = Planetoid(root='./data', name='Cora')
data = dataset[0]

print(data)
print(dataset.num_features)
print(dataset.num_classes)
print(data.num_nodes)
print(data.num_edges)
print(data.train_mask.sum())
print(data.has_self_loops())
print(data.is_undirected())

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
1433
7
2708
10556
tensor(140)
False
True


In [ ]:
class GCNConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # nn.Linear가 내부에 가중치 행렬 W를 저장하고 있는 객체
        # self.W(x) 호출 시 -> x @ W^T 연산 실행 (forward() 자동 호출)
        self.W = nn.Linear(in_channels, out_channels, bias=False)
        

    def forward(self, x, edge_index, num_nodes):
        # Step 1: A_tilde = A + I (자기 자신도 이웃으로 포함)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)

        # Step 2: 각 노드의 degree 계산 후 D_tilde^{1/2} 구하기
        row, col = edge_index # row=source 노드, col=target 노드
        deg = degree(col, num_nodes=num_nodes, dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5) # shape: [N]

        # Step 3: 엣지별 정규화 계수 = d_i^{-1/2} * d_j^{-1/2}, shape: [E]
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Step 4: 선형 변환 XW (모든 노드 feature를 새 차원으로 투영)
        # [N, in_channels] -> [N, out_channels]
        x = self.W(x) # [N, out_channels]
                      # self.W.forward(x)과 동치

        # Step 5: 정규화된 이웃 집계 (D^{-1/2} A D^{-1/2} H)
        out = torch.zeros(num_nodes, x.size(1), device=x.device) # 결과를 담을 빈 행렬 [N, out_channels]
        out.scatter_add_( # src 값들을 index가 가리키는 위치에 더해 넣음 -> 엣지로 연결된 이웃 노드들의 feature를 target 노드에 합산
            dim=0, # out[index[i][j]][j] += src[i][j] 열번호는 그대로, 행 번호만 index가 지정
            # col: target 노드 index [E] -> [E, out_channels]로 확장 (scatter_add는 index, src shape이 동일해야 함)
            index=col.unsqueeze(-1).expand_as(x[row]), 
            # norm [E] -> [E, 1]으로 unsqueeze 후, x[row] [E, out_channels]와 브로드캐스트 곱
            # 각 엣지의 source 노드 feature에 정규화 계수 곱함
            src=norm.unsqueeze(-1) * x[row]
            # out[col] += norm * x[row] <- target 노드에 이웃 feature 합산
        )
        return out
    
            # 브로드캐스트: 크기가 다른 두 배열을 곱할 때, 작은 쪽을 자동으로 큰 쪽에 늘려서 맞춰주는 것
            # a = [2, 3, 4], b = 10 일 때, a * b = [20, 30, 40]

In [7]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index, num_nodes):
        x = self.conv1(x, edge_index, num_nodes)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, num_nodes)
        return F.log_softmax(x, dim=1)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

model = GCN(
    in_channels=dataset.num_features, 
    hidden_channels=16, 
    out_channels=dataset.num_classes,
    dropout=0.5
    ).to(device)
# dataset.num_features: 각 노드가 가진 feature의 길이

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=0.01, 
    weight_decay=5e-4
    )

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.num_nodes)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    out = model(data.x, data.edge_index, data.num_nodes)
    pred = out.argmax(dim=1)
    results = {}
    for split, mask in [('train', data.train_mask),
                        ('val',   data.val_mask),
                        ('test',  data.test_mask)]:
        correct = (pred[mask] == data.y[mask]).sum().item()
        results[split] = correct / mask.sum().item()
    return results

In [9]:
best_val_acc = 0
best_test_acc = 0

for epoch in range(1, 201):
    loss = train()
    accs = test()

    if accs['val'] > best_val_acc:
        best_val_acc = accs['val']
        best_test_acc = accs['test']

    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | "
              f"Train: {accs['train']:.4f} | "
              f"Val: {accs['val']:.4f} | "
              f"Test: {accs['test']:.4f}")

print(f"\n Best Val: {best_val_acc:.4f} | Best Test: {best_test_acc:.4f}")
print(f" Paper target: ~0.815")

Epoch 020 | Loss: 0.3951 | Train: 0.9857 | Val: 0.7700 | Test: 0.7990
Epoch 040 | Loss: 0.0894 | Train: 1.0000 | Val: 0.7740 | Test: 0.7970
Epoch 060 | Loss: 0.0465 | Train: 1.0000 | Val: 0.7720 | Test: 0.8020
Epoch 080 | Loss: 0.0434 | Train: 1.0000 | Val: 0.7640 | Test: 0.8030
Epoch 100 | Loss: 0.0354 | Train: 1.0000 | Val: 0.7720 | Test: 0.7930
Epoch 120 | Loss: 0.0518 | Train: 1.0000 | Val: 0.7740 | Test: 0.7990
Epoch 140 | Loss: 0.0282 | Train: 1.0000 | Val: 0.7740 | Test: 0.8150
Epoch 160 | Loss: 0.0481 | Train: 1.0000 | Val: 0.7660 | Test: 0.7970
Epoch 180 | Loss: 0.0255 | Train: 1.0000 | Val: 0.7660 | Test: 0.8040
Epoch 200 | Loss: 0.0357 | Train: 1.0000 | Val: 0.7640 | Test: 0.7900

 Best Val: 0.7880 | Best Test: 0.8050
 Paper target: ~0.815
